In [5]:
import torch
import torch.nn as nn

# 理论验证：使用 PyTorch 的 Conv2d 验证输出尺寸
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, padding=2, stride=2)
x = torch.randn(1, 3, 32, 32)  # (Batch, Channel, H, W)
out = conv(x)
print(f"PyTorch 实际输出尺寸: {out.shape}")  # torch.Size([1, 16, 16, 16])

# 计算单个像素的乘法次数（不考虑 bias）
kernel_h, kernel_w = 5, 5
in_c = 3
mul_per_pixel = kernel_h * kernel_w * in_c
print(f"单个输出像素点乘次数: {mul_per_pixel}")  # 75

PyTorch 实际输出尺寸: torch.Size([1, 16, 16, 16])
单个输出像素点乘次数: 75


In [6]:
import numpy as np

def manual_maxpool2d(x, kernel_size, stride=2, padding=0):
    """
    x: np.ndarray, shape (C, H, W) 或 (H, W)
    kernel_size: int 或 tuple (kh, kw)
    stride: int 或 tuple
    padding: int
    """
    # 统一格式
    if isinstance(kernel_size, int):
        kh = kw = kernel_size
    else:
        kh, kw = kernel_size
    
    if isinstance(stride, int):
        sh = sw = stride
    else:
        sh, sw = stride
    
    # 1. 处理维度：统一为 (C, H, W)
    if x.ndim == 2:
        x = x[np.newaxis, :, :]  # (1, H, W)
    C, H, W = x.shape
    
    # 2. Padding
    if padding > 0:
        x_pad = np.pad(x, ((0, 0), (padding, padding), (padding, padding)), 
                       mode='constant', constant_values=0)
    else:
        x_pad = x
    H_pad, W_pad = x_pad.shape[1], x_pad.shape[2]
    
    # 3. 计算输出尺寸
    out_h = (H_pad - kh) // sh + 1
    out_w = (W_pad - kw) // sw + 1
    out = np.zeros((C, out_h, out_w))
    
    # 4. 滑动窗口取最大值
    for i in range(out_h):
        for j in range(out_w):
            h_start = i * sh
            w_start = j * sw
            window = x_pad[:, h_start:h_start+kh, w_start:w_start+kw]
            out[:, i, j] = np.max(window, axis=(1, 2))
    
    return out.squeeze()  # 如果原来是2D，则去掉单通道维度

# 测试
x = np.random.randn(3, 32, 32)
out = manual_maxpool2d(x, kernel_size=2, stride=2, padding=0)
print(f"手动池化输出尺寸: {out.shape}")  # (3, 16, 16)

手动池化输出尺寸: (3, 16, 16)


In [7]:
import torch.nn as nn

def count_params(module):
    return sum(p.numel() for p in module.parameters())

C = 64  # 假设通道数

# 模拟 5x5 卷积
conv5x5 = nn.Conv2d(C, C, kernel_size=5, bias=False)
print(f"5x5 参数量: {count_params(conv5x5)}")  # 102400 = 25*64^2

# 模拟 两个 3x3 串联
conv3x3_1 = nn.Conv2d(C, C, kernel_size=3, bias=False)
conv3x3_2 = nn.Conv2d(C, C, kernel_size=3, bias=False)
total = count_params(conv3x3_1) + count_params(conv3x3_2)
print(f"两个3x3总参数量: {total}")  # 73728 = 18*64^2

5x5 参数量: 102400
两个3x3总参数量: 73728


In [8]:
import torch.nn as nn

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super().__init__()
        self.block = nn.Sequential(
            # 普通卷积层
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            # 第一个 1x1 卷积 (通道数保持不变)
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            # 第二个 1x1 卷积 (输出通道数)
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.block(x)

# 测试
block = NiNBlock(in_channels=3, out_channels=96, kernel_size=3, stride=1, padding=1)
x = torch.randn(1, 3, 32, 32)
out = block(x)
print(f"NiN Block 输出尺寸: {out.shape}")  # torch.Size([1, 96, 32, 32])

NiN Block 输出尺寸: torch.Size([1, 96, 32, 32])


In [9]:
import numpy as np

x = np.array([2, 4, 6, 8], dtype=np.float32)
gamma, beta, eps = 2.0, 1.0, 0.0

mean = np.mean(x)
var = np.var(x)  # 注意：np.var 默认总体方差，与题目一致
x_hat = (x - mean) / np.sqrt(var + eps)
y = gamma * x_hat + beta

print(f"均值: {mean}, 方差: {var}")
print(f"BN 输出结果: {y}")  # [-1.683, 0.106, 1.894, 3.683]

均值: 5.0, 方差: 5.0
BN 输出结果: [-1.6832814   0.10557282  1.8944272   3.6832814 ]


In [10]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super().__init__()
        # 主路径：两个 3x3 卷积，每个后面跟 BN
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               padding=1, stride=stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, 
                               padding=1, stride=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 残差连接（捷径）
        if use_1x1conv:
            self.conv_shortcut = nn.Conv2d(in_channels, out_channels, 
                                           kernel_size=1, stride=stride)
            self.bn_shortcut = nn.BatchNorm2d(out_channels)
        else:
            self.conv_shortcut = None
    
    def forward(self, x):
        # 主路径
        y = torch.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        
        # 捷径
        if self.conv_shortcut is not None:
            x = self.bn_shortcut(self.conv_shortcut(x))
        
        # 相加 + 激活
        out = torch.relu(y + x)
        return out

# 测试
block = Residual(in_channels=3, out_channels=64, use_1x1conv=True, stride=2)
x = torch.randn(1, 3, 32, 32)
out = block(x)
print(f"Residual Block 输出尺寸: {out.shape}")  # torch.Size([1, 64, 16, 16])

Residual Block 输出尺寸: torch.Size([1, 64, 16, 16])


In [11]:
from torchvision import transforms

augmentation_pipeline = transforms.Compose([
    # 1. 随机裁剪并缩放至 224x224
    transforms.RandomResizedCrop(size=224, scale=(0.08, 1.0)),
    # 2. 50% 概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    # 3. 颜色抖动（亮度、对比度、饱和度）
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    # 4. 转 Tensor
    transforms.ToTensor()
])

# 模拟使用（假设已有 PIL 图片）
# from PIL import Image
# img = Image.open('test.jpg')
# tensor_img = augmentation_pipeline(img)
# print(tensor_img.shape)  # torch.Size([3, 224, 224])
print("增广 Pipeline 定义完成！")

增广 Pipeline 定义完成！


In [12]:
def calculate_iou(box1, box2):
    """
    box format: [x1, y1, x2, y2]  (左上角, 右下角)
    """
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    # 交集面积
    inter_w = max(0, x2 - x1)
    inter_h = max(0, y2 - y1)
    inter_area = inter_w * inter_h
    
    # 各自面积
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    union_area = area1 + area2 - inter_area
    iou = inter_area / union_area if union_area > 0 else 0
    return iou

A = [10, 10, 50, 50]
B = [30, 30, 70, 70]
iou_val = calculate_iou(A, B)
print(f"IoU 准确值: {iou_val}")  # 0.14285714285714285
print(f"分数表示: 1/7") 

IoU 准确值: 0.14285714285714285
分数表示: 1/7


In [ ]:
import torch
import torch.nn.functional as F

def label_smoothing_loss(pred, target, epsilon=0.1, num_classes=None):
    """
    pred: 模型输出 logits, shape (N, K)
    target: 真实标签索引, shape (N,)
    """
    if num_classes is None:
        num_classes = pred.size(1)
    
    # 计算标准交叉熵损失 (用于 debug 或 basline)
    # log_probs = F.log_softmax(pred, dim=-1)
    # loss = -log_probs.gather(1, target.unsqueeze(1)).squeeze()
    
    # 1. 构造平滑标签矩阵
    smooth_label = torch.full_like(pred, epsilon / (num_classes - 1))
    smooth_label.scatter_(1, target.unsqueeze(1), 1 - epsilon)
    
    # 2. 计算 log_softmax
    log_probs = F.log_softmax(pred, dim=-1)
    
    # 3. 交叉熵： -sum(label * log_prob)
    loss = -(smooth_label * log_probs).sum(dim=-1).mean()
    return loss

# 测试
num_classes = 5
pred = torch.randn(4, num_classes)  # 4个样本，5分类
target = torch.tensor([1, 0, 3, 2])  # 真实标签

loss = label_smoothing_loss(pred, target, epsilon=0.1)
print(f"标签平滑后的 Loss: {loss.item():.4f}")

# 对比标准交叉熵（用于观察平滑效果）
standard_loss = F.cross_entropy(pred, target)
print(f"标准交叉熵 Loss: {standard_loss.item():.4f}")

标签平滑后的 Loss: 2.0025
标准交叉熵 Loss: 2.0080
